In [84]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import optuna
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from collections import defaultdict

In [103]:
SEED = 42
np.random.seed(SEED)

def suffix_generator():
    yield ''
    i = 1
    while True:
        yield f'_{i}'
        i += 1


In [104]:
df = pd.read_excel('database/carbonation-data.xlsx', sheet_name="Carbonatation depth CRA")

In [105]:
counts = defaultdict(suffix_generator)
df.columns = [col + next(counts[col]) for col in df.columns]

In [106]:
df.columns

Index([' ', 'Unnamed: 1', 'Apparent Particle Density ',
       'Saturated Surface Dry- Particle Density ',
       'Oven-dry particles density', 'Oven-dry particles density.1',
       'OD part dens index', 'OD part dens index.1', 'Weighted density ',
       'Bulk density', 'Bulk dens index', 'Bulk density.1',
       'Fine Agg water abs and density', 'Replacement level',
       'Water absoprtionGravel 1', 'Water absorptionGravel 2', 'WA for CNA',
       'WA coeff for CNA', 'Water absorption ', 'WA coeff for CRA',
       'ShapeIndexGravel 1', 'ShapeIndexGravel 2', 'SI for CNA',
       'SI coeff for CNA', 'Shape index RA', 'SI coeff for CRA',
       'Los Angeles wear Gav1', 'Los Angeles wear Gav2', 'LA for CNA',
       'LA coeff for CNA', 'Los Angeles wear', 'LA coeff for CRA',
       'Blast Furnace Slag', 'Metakaolin', 'Fly ash content',
       'Limestone filler', 'Cement Content', 'Cement Content.1', 'Silica fume',
       'Water content', 'Water content.1', 'Effective water-to-bider rati

In [112]:
cols = ["Water absorption ", "Effective water-to-bider ratio", "Fine aggregate  content", "Gravel content.1", "RA content .1", "Superplasticizer", "Carbon concentration", "Exposure time", "Carbonation depth", 'Cube compressive strength ']

In [115]:
df = df[cols]
df = df.apply(pd.to_numeric, errors="coerce").astype(float)
df = df.replace("-", np.nan)
df = df.dropna()

In [117]:
df

,Water absorption,Effective water-to-bider ratio,Fine aggregate content,Gravel content.1,RA content .1,Superplasticizer,Carbon concentration,Exposure time,Carbonation depth,Cube compressive strength
2,0.0,0.510000,357.655,981.7080,0.00000,0.0,5.0,7.0,2.7,53.9
3,8.6,0.520000,357.655,883.5372,78.83070,0.0,5.0,7.0,2.7,54.1
4,8.6,0.530000,357.655,736.2810,197.07675,0.0,5.0,7.0,3.9,48.9
5,8.6,0.530000,357.655,490.8540,394.15350,0.0,5.0,7.0,3.5,46.2
6,8.6,0.530000,357.655,0.0000,788.30700,0.0,5.0,7.0,5.1,35.3
...,...,...,...,...,...,...,...,...,...,...
753,3.6,0.439429,859.000,0.0000,943.00000,3.5,5.0,56.0,4.1,68.7
754,3.9,0.442000,859.000,0.0000,985.00000,3.5,5.0,56.0,4.2,66.9
755,0.0,0.430000,859.000,1027.0000,0.00000,3.5,5.0,91.0,5.0,72.6
756,3.6,0.439429,859.000,0.0000,943.00000,3.5,5.0,91.0,6.3,68.7


In [118]:
df.to_csv("cube_strength_carbonation_depth_data.csv")

In [20]:
df = pd.read_excel('database/carbonation-data.xlsx', sheet_name="Preprocessed data")
df.drop(df.columns[df.columns.str.contains('unnamed', case=False)], axis=1, inplace=True)
df.rename(columns={df.columns[0]: 'Paper ID'}, inplace=True)
df.iloc[:, 0] = df.iloc[:, 0].ffill()
df.tail()

,Paper ID,RAWA,WBR,FAC,GC,RAC,SP,CC,T,CD
677,[23],3.9,0.442000,859.0,0.0,985.0,3.5,5.0,28.0,2.1
678,[23],3.6,0.439429,859.0,0.0,943.0,3.5,5.0,56.0,4.1
679,[23],3.9,0.442000,859.0,0.0,985.0,3.5,5.0,56.0,4.2
680,[23],3.6,0.439429,859.0,0.0,943.0,3.5,5.0,91.0,6.3
681,[23],3.9,0.442000,859.0,0.0,985.0,3.5,5.0,91.0,6.2


In [3]:
def variance_accounted_for(y_true, y_pred):
    """Calculates Variance Accounted For (VAF) in percentage."""
    var_y = np.var(y_true)
    var_res = np.var(y_true - y_pred)
    return (1 - (var_res / var_y)) * 100

def mean_bias_error(y_true, y_pred):
    """Calculates Mean Bias Error (MBE)."""
    return np.mean(y_pred - y_true)

def mean_absolute_percentage_error_custom(y_true, y_pred):
    """Calculates MAPE in percentage."""
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def evaluate_model(y_true, y_pred):
    """Returns all 6 evaluation metrics specified in the paper."""
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "MAPE": mean_absolute_percentage_error_custom(y_true, y_pred),
        "VAF": variance_accounted_for(y_true, y_pred),
        "MBE": mean_bias_error(y_true, y_pred)
    }

In [16]:
raw_df.columns


Index([' ', 'Unnamed: 1', 'RAWA', 'WBR', 'FAC', 'GC', 'RAC', 'SP', 'CC', 'T',
       'CD'],
      dtype='object')